# Setup: functions to generate populations with its respective initial conditions and a runnable simulation object.
- Contents:
    - setup_mixed: create population of individuals and households objects.
        - Fill (populate) population objects.
        - Link individuals within households and communities.
    - setup_sim: generate runnable simulation object.

1. Call input code files: model and analysis.

In [7]:
include("agents.jl")
include("model.jl")

update_agents! (generic function with 1 method)

2. *setup_mixed* function: create population of individuals and households objects based on the parameters specified when running the simulation.
- Function field should consists of an *Params* object. We will use the previosly defined struct *Params* to define this object.
- Creation of a population of individuals (*pop*) and households (*pop_hh*) objects. 
    - Create *n* and *n_hh* number of agents and households, respectively. These numbers are model parameters. See *Params* properties defined in "agents.jl".
- Fill households with agents. 
- Define agents' families by household membership as agents'. Remember that agents' families is a property of agents (see Person struct in "agents.jl".).
- Define households' vulnerability.
- Define agents' communities: assign aggents to a community list, based on a probability of contact (*p_contact*). 
- Define agents' income. Based from empirical income distribution (e.g. Colombian household survey).
- Define agents' potential gain and losses from migration.
- Output: *pop* and *pop_hh*

In [9]:
function setup_mixed(par :: Params)
    #Create agents and households
    pop = [ Person(i) for i = 1:par.n ]
    pop_hh = [ Household() for i = 1:par.n_hh ]
    ##Households
    # Assign agents to a random household.
    for p in pop
      push!(rand(pop_hh).members, p)
    end
    #Copy hh members to agents family. (A bit messy code - to be improved)
    for hh in pop_hh
      for j in eachindex(hh.members)
        for i in eachindex(hh.members)
          push!(hh.members[j].family, hh.members[i])
        end
      end
    end
    for hh in pop_hh
      hh.vuln = rand()
    end
    #Households
    #Create agents community list
    for i in eachindex(pop)
        for j in i+1:length(pop)
            if rand() < par.p_contact
                push!(pop[i].community, pop[j])
                push!(pop[j].community, pop[i])
            end
        end
    end
    for p in pop
        p.ainc = sample(par.empInc, par.weights)
    end
    for p in pop
        p.gain = rand()
        p.loss = rand()
    end
    pop, pop_hh
end

setup_mixed (generic function with 1 method)

3. Create a simulation object containing the populations of interest.
- Function fields is also a *Params* object.
- *Params* used are a random seed and number of migrants.
- Output: a simulation object containing a population of individuals and a population of households.
- Note *setup_sim* uses previous function, *setup_mixed*.

In [10]:
function setup_sim(;par :: Params)
    Random.seed!(par.seed)
    # Parameters as the field to create agents and households
    # Function creates two objects: pop and pop_hh
    pop, pop_hh = setup_mixed(par)
    # Only create one object sim with pop and pop_hh?
    sim = Simulation(pop, pop_hh)
    for i in 1:par.nmig
        # number of migrants before simulation begins. Default 0.
        sim.pop[i].status = migrant
    end
    sim
end

setup_sim (generic function with 1 method)